# Loading the NP-Ultra 2024 dataset with `from_dict`

This dataset is the flattened figshare release of Ye, Z., Shaw, T. L., et al. (2024),
*Ultra-dense electrode recordings reveal cell-type-specific waveforms across the mouse brain*.
It is a single concatenation of curated clusters across 4 mice, 8 sessions, one NP-Ultra
probe each.

It is not a Phy or Kilosort sorter directory, so `from_phy` and `from_kilosort` do not
apply. The correct entry point is `neurocomplexity.io.from_dict`, which builds a
`SpikeRecording` from an in-memory `{unit_id: spike_times_seconds}` mapping. This notebook
shows the full path: split the global arrays into one session, assemble per-unit metadata,
and call `from_dict`.

Set `DATA_DIR` below to your local copy of the dataset.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import neurocomplexity as nc
from neurocomplexity.io import from_dict

DATA_DIR = Path("mouse_waveform_dataset_2024")  # set to your local copy
assert DATA_DIR.exists(), DATA_DIR
sorted(p.name for p in DATA_DIR.iterdir())

## 1. File inventory

| File | Shape | Meaning |
|------|-------|---------|
| `spikes.times.npy` | `(1, N_spikes)` float64 | Spike times in seconds. Each session restarts near 0 s; the full array is not one continuous timeline. |
| `spikes.clusters.npy` | `(1, N_spikes)` float64 | Per-spike cluster id. Values are 1-indexed row positions in `all_clusters_table.csv` (Python row = `cluster_id - 1`). |
| `clusters.waveforms.npy` | `(4666, 82, 384)` float64 | Mean waveform per unit: 82 samples at 30 kHz across 384 channels. |
| `channels.xcoords.npy` / `channels.ycoords.npy` | `(384, 1)` | Channel positions in micrometres. |
| `clusters.CCF_APDVLR.npy` | `(N_units, 3)` | Allen CCF coordinates (AP, DV, LR) in micrometres. |
| `clusters.acronym.csv` | `(N_units, 10)` | Hierarchical anatomy path. |
| `all_clusters_table.csv` | `(N_units, 31)` | Per-unit metadata: `MouseID, ExpDate, en, clusterID, SpikeN, duration, sign, footprint, CCFAcronym, keep, artefacts, ...`. |

In [ ]:
# mmap_mode keeps the 11+ GB waveform array on disk instead of in RAM.
spike_times = np.load(DATA_DIR / "spikes.times.npy", mmap_mode="r").ravel()
spike_clu   = np.load(DATA_DIR / "spikes.clusters.npy", mmap_mode="r").ravel().astype(np.int64)
waveforms   = np.load(DATA_DIR / "clusters.waveforms.npy", mmap_mode="r")
xc          = np.load(DATA_DIR / "channels.xcoords.npy").ravel()
yc          = np.load(DATA_DIR / "channels.ycoords.npy").ravel()
ccf         = np.load(DATA_DIR / "clusters.CCF_APDVLR.npy")
units_tbl   = pd.read_csv(DATA_DIR / "all_clusters_table.csv")

print("spike_times:", spike_times.shape, spike_times.dtype)
print("spike_clu  :", spike_clu.shape, "min", spike_clu.min(), "max", spike_clu.max())
print("waveforms  :", waveforms.shape)
print("units_tbl  :", units_tbl.shape)
units_tbl.head(3)

## 2. Discover sessions

A session is a unique `(MouseID, ExpDate, en)` triple. `neurocomplexity` expects one
`SpikeRecording` per session, so list the sessions and pick one.

In [ ]:
sessions = (
    units_tbl.groupby(["MouseID", "ExpDate", "en"])
             .agg(n_units=("clusterID", "size"), total_spikes=("SpikeN", "sum"))
             .reset_index()
)
sessions

In [ ]:
sess_row = sessions.sort_values("n_units", ascending=False).iloc[0]
MOUSE, DATE, EN = sess_row["MouseID"], sess_row["ExpDate"], int(sess_row["en"])
print("session:", MOUSE, DATE, "epoch", EN, "n_units", int(sess_row["n_units"]))

## 3. Slice the global arrays to one session

The index relationship that ties the arrays together:

- `units_tbl` row `i` (0-based) corresponds to waveform row `i` and to `spike_clu` value `i + 1`.
- So the spike mask for a session is `np.isin(spike_clu, session_rows + 1)`.

After masking, cluster ids are re-keyed to a contiguous `0..n_units-1` range so they line
up with the metadata table built in the next step.

In [ ]:
sess_mask  = ((units_tbl["MouseID"] == MOUSE)
              & (units_tbl["ExpDate"] == DATE)
              & (units_tbl["en"] == EN))
sess_units = units_tbl[sess_mask].copy()
sess_rows  = sess_units.index.to_numpy()   # 0-based row positions
sess_cids  = sess_rows + 1                  # 1-based ids used in spike_clu

spike_keep = np.isin(spike_clu, sess_cids)
st_sess    = np.asarray(spike_times[spike_keep], dtype=np.float64)
cl_sess    = np.asarray(spike_clu[spike_keep],   dtype=np.int64)

# Re-key cluster ids to 0..n_units-1 within the session.
cid_to_local = {int(c): i for i, c in enumerate(sess_cids)}
cl_local     = np.fromiter((cid_to_local[c] for c in cl_sess),
                           dtype=np.int64, count=cl_sess.size)

duration_s = float(st_sess.max() + 1.0)  # last spike + 1 s pad
print("session spikes:", st_sess.size)
print("session units :", sess_rows.size)
print("duration      :", round(duration_s, 1), "s")

## 4. Per-unit metadata

`from_dict` accepts an optional `unit_metadata` DataFrame. It must contain an `id` column
matching the keys of the spike mapping. Columns that `neurocomplexity` recognises
(`quality`, `firing_rate`, `peak_channel`, `brain_area`) let downstream filters and
population grouping work without extra steps; any other columns are carried through verbatim.

The peak channel is the channel of largest peak-to-peak amplitude, computed on baseline-
subtracted waveforms (first 10 samples used as baseline).

In [ ]:
sess_wfs = np.asarray(waveforms[sess_rows], dtype=np.float64)   # (n_units, 82, 384)
sess_wfs -= sess_wfs[:, :10, :].mean(axis=1, keepdims=True)     # baseline subtract
ptp = sess_wfs.ptp(axis=1)                                      # (n_units, 384)
peak_channel = ptp.argmax(axis=1)
peak_amp_uv  = ptp.max(axis=1)

spike_counts = np.bincount(cl_local, minlength=sess_rows.size)
firing_rate  = spike_counts / duration_s

unit_meta = pd.DataFrame({
    "id":            np.arange(sess_rows.size, dtype=np.int64),
    "peak_channel":  peak_channel.astype(np.int64),
    "peak_amp_uv":   peak_amp_uv,
    "firing_rate":   firing_rate,
    "spike_count":   spike_counts,
    "brain_area":    sess_units["CCFAcronym"].astype("string").to_numpy(),
    "ccf_ap": ccf[sess_rows, 0], "ccf_dv": ccf[sess_rows, 1], "ccf_ml": ccf[sess_rows, 2],
    "waveform_sign": sess_units["sign"].to_numpy(),
    "duration_ms":   sess_units["duration"].to_numpy(),
    "artefacts":     sess_units["artefacts"].to_numpy(),
    "keep_curated":  sess_units["keep"].to_numpy(),
})

# Quality gate: the release is pre-curated (keep==1 for every row), so the real cut is
# artefacts==0 plus a minimum firing rate.
unit_meta["quality"] = np.where(
    (unit_meta["artefacts"] == 0)
    & (unit_meta["keep_curated"] == 1)
    & (unit_meta["firing_rate"] >= 0.1),
    "good", "noise",
)
unit_meta.head()

## 5. Build a `SpikeRecording`

`from_dict(spike_times_by_unit, duration, unit_metadata=None, sampling_rate=None, hint="dict")`

- `spike_times_by_unit`: `{unit_id: spike_times_seconds}`. Keys must match `unit_metadata['id']`.
- `duration`: recording length in seconds. Rate and avalanche statistics need it to define the
  final time bin, so set it explicitly rather than inferring from the last spike.
- `unit_metadata`: optional per-unit DataFrame (the table from step 4).
- `sampling_rate`: source sampling rate in Hz, stored for provenance (30 kHz here).
- `hint`: short label recorded in the recording's provenance for later identification.

In [ ]:
order  = np.argsort(cl_local, kind="stable")
split  = np.searchsorted(cl_local[order], np.arange(1, sess_rows.size))
groups = np.split(st_sess[order], split)
spike_times_by_unit = {int(i): np.sort(g) for i, g in enumerate(groups)}

rec = from_dict(
    spike_times_by_unit,
    duration=duration_s,
    unit_metadata=unit_meta,
    sampling_rate=30_000.0,
    hint="npultra_2024:" + str(MOUSE) + "_" + str(DATE) + "_e" + str(EN),
)
print(rec)
rec.units.head()

## 6. Preprocessing checks

Before analysis, confirm spike times are sorted and in range, drop low-rate and artefact
units, and check that peak channels fall on the probe shank rather than at the edges.

In [ ]:
# Spike times are sorted and within [0, duration].
assert np.all(np.diff(rec.spike_times) >= 0)
assert rec.spike_times.min() >= 0 and rec.spike_times.max() <= rec.duration

rec_clean = rec.filter_units(quality=["good"])
print("good units kept:", len(rec_clean.units), "/", len(rec.units))
rec_clean.units["brain_area"].value_counts().head(10)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))

ax[0].hist(rec_clean.units["firing_rate"], bins=40, color="#3b6")
ax[0].set(xlabel="firing rate (Hz)", ylabel="units", title="firing-rate distribution")

ax[1].scatter(xc[rec_clean.units["peak_channel"]], yc[rec_clean.units["peak_channel"]],
              s=10, alpha=0.5)
ax[1].set(xlabel="x (um)", ylabel="y (um)", title="peak-channel positions")

t_ms = np.arange(82) / 30.0  # 30 kHz -> ms
for _, row in rec_clean.units.head(6).iterrows():
    pc = int(row["peak_channel"])
    ax[2].plot(t_ms, sess_wfs[int(row["id"]), :, pc], lw=1, label=row["brain_area"])
ax[2].set(xlabel="time (ms)", ylabel="uV", title="peak-channel waveforms")
ax[2].legend(fontsize=7)

plt.tight_layout(); plt.show()

## 7. Smoke-test an analysis

If a standard analysis runs end-to-end, the recording is structurally valid for the rest of
the package. This is a wiring check on default parameters, not a scientific result. Group
units into populations by brain area first, then run `stationarity`.

In [ ]:
rec_clean = rec_clean.with_populations(by="brain_area", on_unassigned="drop")
print("populations:", len(rec_clean.populations))

result = nc.analysis.stationarity(rec_clean)
print(result)

## 8. Reusable loader

The steps above wrapped in a function so the other sessions are one call away. Drop it into
`examples/` or `neurocomplexity/io/npultra_2024.py` if you want it on the public API.

In [ ]:
def load_npultra_2024_session(data_dir, mouse_id, exp_date, en=1, min_rate=0.1):
    """Build a SpikeRecording for one session of the NP-Ultra 2024 figshare dataset."""
    data_dir = Path(data_dir)
    spike_times = np.load(data_dir / "spikes.times.npy", mmap_mode="r").ravel()
    spike_clu   = np.load(data_dir / "spikes.clusters.npy", mmap_mode="r").ravel().astype(np.int64)
    waveforms   = np.load(data_dir / "clusters.waveforms.npy", mmap_mode="r")
    ccf         = np.load(data_dir / "clusters.CCF_APDVLR.npy")
    units_tbl   = pd.read_csv(data_dir / "all_clusters_table.csv")

    mask = ((units_tbl["MouseID"] == mouse_id)
            & (units_tbl["ExpDate"] == exp_date)
            & (units_tbl["en"] == en))
    sess_units = units_tbl[mask]
    if sess_units.empty:
        raise ValueError("no session " + str(mouse_id) + " " + str(exp_date) + " e" + str(en))
    rows = sess_units.index.to_numpy()
    cids = rows + 1

    keep = np.isin(spike_clu, cids)
    st   = np.asarray(spike_times[keep], dtype=np.float64)
    cl   = np.asarray(spike_clu[keep],   dtype=np.int64)
    remap = {int(c): i for i, c in enumerate(cids)}
    cl_local = np.fromiter((remap[c] for c in cl), dtype=np.int64, count=cl.size)
    dur = float(st.max() + 1.0)

    wfs = np.asarray(waveforms[rows], dtype=np.float64)
    wfs -= wfs[:, :10, :].mean(axis=1, keepdims=True)
    ptp = wfs.ptp(axis=1)
    counts = np.bincount(cl_local, minlength=rows.size)

    meta = pd.DataFrame({
        "id":           np.arange(rows.size, dtype=np.int64),
        "peak_channel": ptp.argmax(axis=1).astype(np.int64),
        "peak_amp_uv":  ptp.max(axis=1),
        "firing_rate":  counts / dur,
        "spike_count":  counts,
        "brain_area":   sess_units["CCFAcronym"].astype("string").to_numpy(),
        "ccf_ap": ccf[rows, 0], "ccf_dv": ccf[rows, 1], "ccf_ml": ccf[rows, 2],
        "waveform_sign": sess_units["sign"].to_numpy(),
        "duration_ms":   sess_units["duration"].to_numpy(),
        "artefacts":     sess_units["artefacts"].to_numpy(),
        "keep_curated":  sess_units["keep"].to_numpy(),
    })
    meta["quality"] = np.where(
        (meta["artefacts"] == 0) & (meta["keep_curated"] == 1) & (meta["firing_rate"] >= min_rate),
        "good", "noise",
    )

    order  = np.argsort(cl_local, kind="stable")
    split  = np.searchsorted(cl_local[order], np.arange(1, rows.size))
    groups = np.split(st[order], split)
    spikes = {int(i): np.sort(g) for i, g in enumerate(groups)}

    return from_dict(
        spikes, duration=dur, unit_metadata=meta, sampling_rate=30_000.0,
        hint="npultra_2024:" + str(mouse_id) + "_" + str(exp_date) + "_e" + str(en),
    )


rec2 = load_npultra_2024_session(DATA_DIR, MOUSE, DATE, EN)
print(rec2)

## 9. Caveats

- **Sessions are not concatenated in time.** Each session's spike times restart near 0 s. Do
  not run cross-session analyses on the un-split arrays.
- **Index off-by-one.** `spike_clu` is 1-based (MATLAB origin); Python row index = `cluster_id - 1`.
  The loader hides this from callers.
- **Waveform sign.** `sign == 1` is trough-dominant, `sign == 0` is peak-dominant. Features
  such as trough-to-peak duration assume trough-dominant, so branch on this column.
- **`keep` is constant 1** in this release; use `artefacts` plus a firing-rate cut as the real
  quality gate.
- **CCF coordinates are in micrometres**, not the 25-um voxel space used by the MATLAB example.